# Credit Card Default Data

### Description:
A simulated data set containing information on ten thousand customers. The aim here is to predict which customers will default on their credit card debt.

### Usage
Default

### Format
A data frame with 10000 observations on the following 4 variables.

*default:*
A factor with levels No and Yes indicating whether the customer defaulted on their debt

*student:*
A factor with levels No and Yes indicating whether the customer is a student

*balance:*
The average balance that the customer has remaining on their credit card after making their monthly payment

*income:*
Income of customer

### Source:
Simulated data

### References
James, G., Witten, D., Hastie, T., and Tibshirani, R. (2013) An Introduction to Statistical Learning with applications in R, www.StatLearning.com, Springer-Verlag, New York

In [ ]:
# Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns # for making plots with seaborn
color = sns.color_palette()
import sklearn.metrics as metrics

import warnings
warnings.filterwarnings("ignore")

#!pip install imblearn

Let us now go ahead and read the dataset and check the first five rows of the dataset.

#### Importing the dataset

In [ ]:
Default = pd.read_csv('Default.csv')

#Glimpse of Data
Default.head()

#### First, let us check the number of rows (observations) and the number of columns (variables).

In [ ]:
print('The number of rows (observations) is',Default.shape[0],'\n''The number of columns (variables) is',Default.shape[1])

#### Now, let us check the basic measures of descriptive statistics for the continuous variables.

In [ ]:
Default.describe()

#### Univariate Analysis: Balance & Income variables

In [ ]:
plt.figure(figsize = (15, 5))
plt.subplot(1,2,1)
sns.boxplot(y = Default['balance'])

plt.subplot(1,2,2)
sns.boxplot(y = Default['income'])
plt.show()

#### Univariate Analysis: Student & Default variables

In [ ]:
plt.figure(figsize = (15, 5))
plt.subplot(1,2,1)
sns.countplot(Default['student'])

plt.subplot(1,2,2)
sns.countplot(Default['default'])
plt.show()

#### Now, let us check the basic measures of descriptive statistics for the categorical variables.

In [ ]:
Default["student"].value_counts()

In [ ]:
Default["default"].value_counts()

#### Checking proportion of default

In [ ]:
Default["default"].value_counts(normalize = True)

Data seems highly imbalanced

#### Bivariate Analysis: Default Vs. other variables

In [ ]:
plt.figure(figsize = (15, 5))
plt.subplot(1,2,1)
sns.boxplot(Default['default'], Default['balance'])

plt.subplot(1,2,2)
sns.boxplot(Default['default'], Default['income'])
plt.show()

##### Inference: 
Defaulters seem to have higher outstanding balance compared non-defaulters.
Defaulters' income seems lower compared to non-defaulters.

In [ ]:
pd.crosstab(Default['student'], Default['default'], normalize = 'index').round(2)

#### Check for correlation between independent variables

In [ ]:
sns.heatmap(Default[['balance', 'income']].corr(), annot = True)
plt.show()

#### Check for missing values

In [ ]:
Default.isnull().sum()

There are no missing values in the dataset.

#### Treating outliers present in the 'balance' variable

In [ ]:
Q1, Q3 = Default['balance'].quantile([.25, .75])
IQR = Q3 - Q1
LL = Q1 - 1.5*(IQR)
UL = Q3 + 1.5*(IQR)

In [ ]:
df = Default[Default['balance'] > UL]

In [ ]:
df['default'].count()

In [ ]:
df['default'].value_counts(normalize = True)

Note: We have 31 outliers in total, of which 84%(approx 26 in number) are with a default status 1. As such we have only 333 default instances in our data therefore dropping these 26 i.e. 8% will not be a good option. 

In [ ]:
Default['balance'] = np.where(Default['balance'] > UL, UL, Default['balance'] )

In [ ]:
sns.boxplot(y = Default['balance'])
plt.show()

Outliers have been replaced

# Start of Credit Risk Modelling PD 

#### Transforming categorical variables into 1 & 0 using pandas get_dummies

In [ ]:
#from sklearn import LabelEncoder
Default = pd.get_dummies(Default, drop_first = True)

#### Getting Top 5 rows

In [ ]:
Default.head()

Relabeling the columns as per original names

In [ ]:
Default.columns = ['balance', 'income', 'default', 'student']

In [ ]:
Default.head()

#### Partitioning the data into train and test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = Default.drop('default', axis = 1)
y = Default['default']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 21, stratify = y)

In [ ]:
print(X_train.shape)
print(X_test.shape)

##### Why stratify = y?
Please note, because this data is highly imbalanced and could possibly result into different proportions in the y variable between train and test set.

In [ ]:
print(y_train.value_counts(normalize = True).round(2))
print(' ')
print(y_test.value_counts(normalize = True).round(2))

#### Treating target imbalance using SMOTE

In [ ]:
#!pip install imblearn

In [ ]:
from imblearn.over_sampling import SMOTE 
sm = SMOTE(random_state=33, sampling_strategy = 0.75)
X_res, y_res = sm.fit_sample(X_train, y_train)

In [ ]:
Default_smote = pd.concat([X_res, y_res], axis = 1)

In [ ]:
Default.groupby('default').mean()

In [ ]:
Default_smote.groupby('default').mean()

# Model Building using Logistic Regression for 'Probability at default'

## The equation of the Logistic Regression by which we predict the corresponding probabilities and then go on predict a discrete target variable is
# y = $\frac{1}{1 + {e^{-z}}}$

### Note: z  = $\beta_0$ +${\sum_{i=1}^{n}(\beta_i  X_1)}$

#### Now, Importing statsmodels modules

In [ ]:
import statsmodels.formula.api as SM

#### Creating logistic regression equation & storing it in f_1

model = SM.logit(formula=’Dependent Variable ~ Σ𝐼𝑛𝑑𝑒𝑝𝑒𝑛𝑑𝑒𝑛𝑡 𝑉𝑎𝑟𝑖𝑎𝑏𝑙𝑒𝑠 (𝑘)’
               data = ‘Data Frame containing the required values’).fit()

In [ ]:
train = pd.concat([X_train, y_train], axis = 1)
train_smote = pd.concat([X_res, y_res], axis = 1)
test = pd.concat([X_test, y_test], axis = 1)

In [ ]:
f_1 = 'default ~ student + balance + income'

#### Fitting the logistic regression model on imbalanced data

In [ ]:
model_1 = SM.logit(formula = f_1, data= train).fit()

#### Checking the parameters

In [ ]:
model_1.summary()

#### Validating the model on train set 

In [ ]:
y_pred_train = np.where(model_1.predict(train) > 0.5, 1, 0)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
print(confusion_matrix(y_train, y_pred_train))

In [ ]:
print(classification_report(y_train, y_pred_train))

#### Validating the model on test set 

In [ ]:
y_pred_test = np.where(model_1.predict(test) > 0.5, 1, 0)

In [ ]:
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
print(classification_report(y_test, y_pred_test))

Model is not overfitting but the recall is really poor.

#### Fitting the logistic regression model on balanced data

In [ ]:
model_2 = SM.logit(formula = f_1, data= train_smote).fit()

In [ ]:
model_2.summary()

#### Validating on resampled train set

In [ ]:
y_pred_train_smote = np.where(model_2.predict(train_smote) > 0.5, 1, 0)

In [ ]:
print(confusion_matrix(y_res, y_pred_train_smote))

In [ ]:
print(classification_report(y_res, y_pred_train_smote))

#### Validating on test set

In [ ]:
y_pred_test_smote = np.where(model_2.predict(test) > 0.5, 1, 0)

In [ ]:
print(confusion_matrix(y_test, y_pred_test_smote))

In [ ]:
print(classification_report(y_test, y_pred_test_smote))

## Conclusion
We can see that we get better recall value after balancing the data but precision is now a problem. This trade-off between recall and precision can be approached by adjusting the threshold. Note that at present we are using a threshold of 0.5.

Also, learners might want to try other classification models(particularly the ones that are not sensitive to outliers) and perform a comparison between different models.  

## END